In [1]:
import asyncio
import httpx
import logging
import json
import sys
import os
from pathlib import Path
import time

In [4]:
import requests

In [71]:
from autogen_core.code_executor import CodeBlock,CodeExecutor
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from autogen_ext.code_executors.jupyter import JupyterCodeExecutor
import tempfile
from pathlib import Path
import venv
import asyncio
from autogen_core import CancellationToken
async def LocalCodeExecutor(codeblock_list,env=None,filedir='/oper/ch/autogen'):
    work_dir = Path(filedir)
    work_dir.mkdir(exist_ok=True)
    if not env:
        venv_dir = work_dir / ".venv"
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_builder.create(venv_dir)
        venv_context = venv_builder.ensure_directories(venv_dir)
    else:
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_context = venv_builder.ensure_directories(env)
    local_executor = LocalCommandLineCodeExecutor(work_dir=work_dir, virtual_env_context=venv_context)
    try:
        result = await local_executor.execute_code_blocks(
            code_blocks=codeblock_list,
            cancellation_token=CancellationToken(),)
        return result.output
    except Exception as e:
        return f"错误问题: {e}"

In [3]:
#sys.path.append('/oper/work/endian/intelligent_agent')

In [53]:
user_api_key={"api-key": "chenhao"}#hao

In [51]:
user_api_key={"api-key": "wangendian"}#endian

In [54]:
response = requests.post(
    "http://localhost:8005/tabs",
    headers=user_api_key,
    json={"provider": "claude"}
)
# 获取tab_id用于后续操作
tab_id = response.json()#["tab_id"]
print(tab_id)

{'status': 'success', 'message': '已有claude标签页', 'tab_id': 'a1b05d0c-b7a0-42e3-bf1f-9a13e302a00c', 'provider': 'claude', 'title': 'Claude', 'url': 'https://claude.ai/new'}


In [196]:
response = requests.post(
    "http://localhost:8005/tabs/chatgpt/screenshot",
    headers=user_api_key,
    json={"provider": "chatgpt"}
)

In [197]:
response.json()

{'status': 'success',
 'screenshot_path': 'screenshots/screenshot_1743157130.png'}

In [45]:
def send_wechat_message():
    processed_params = {
        "contact_name": "陈浩",
        "message": "测试微信数据接口 "
    }
    response = requests.post(
        "http://localhost:8003/tools/wechat/search_and_send",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [121]:
def check_tax():
    processed_params = {
        "city": "深圳",
        "start_date": "2025-01-01",
        "end_date": "2025-04-01"
    }
    response = requests.post(
        "http://localhost:8003/tools/tax/select_city",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [122]:
response=check_tax()
print(response)

{}


In [201]:
# 发送消息给Claude
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": """这段代码llm_page_analyzer是用于不同llm网站都能通用的解析并生成页面元素selectors和extract_page_content。
        经过测试可以试用于claude，但是对于chatgpt，发现无法extract_page_content，问题如下：
        Extracting page content...
        Extracting content from ChatGPT chat interface...
        JavaScript execution error: Cannot find main content area
        请根据代码已经生成的网站元素chatgpt_structure.txt，扩展extract_page_content的通用能力，能不仅试用于claude,还试用于chatgpt,gemini,qwen,deepseek等llm网站。
        请生成完整的extract_page_content代码。 
""",
        "file_paths":["/oper/work/endian/intelligent_agent/page_analyzer/llm_page_analyzer.py",
                     ],
        "new_chat": True
    }
)
#print(response.json())

In [202]:
print(response.json()[]

{'id': 'chatcmpl-087d1995-4ec0-472d-9d1e-abb005596832', 'created': 1743158487, 'model': 'Claude 3.7 Sonnet', 'messages': [{'role': 'user', 'content': '这段代码llm_page_analyzer是用于不同llm网站都能通用的解析并生成页面元素selectors和extract_page_content。         经过测试可以试用于claude，但是对于chatgpt，发现无法extract_page_content，问题如下：         Extracting page content...         Extracting content from ChatGPT chat interface...         JavaScript execution error: Cannot find main content area         请根据代码已经生成的网站元素chatgpt_structure.txt，扩展extract_page_content的通用能力，能不仅试用于claude,还试用于chatgpt,gemini,qwen,deepseek等llm网站。         请生成完整的extract_page_content代码。'}, {'role': 'assistant', 'content': {'response': ["I'll help create a more robust and universal extract_page_content function that can work across different LLM interfaces including ChatGPT, Gemini, Qwen, DeepSeek, and others. This updated version will use more flexible selectors and fallback methods to extract conversation content from various LLM web interfaces.\nLet me write a 

In [203]:
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": "continue",
        "file_paths":None,
        "new_chat": False
    }
)

In [205]:
response.json()['messages'][-1]['content']['codeBlocks'][-1]

{'code': 'const rect = el.getBoundingClientRect();\n                        return text.length > 50 && // Has substantial text\n                              rect.width > 100 && // Has reasonable width\n                              !el.querySelector(\'pre\') && // Not just code\n                              rect.height > 30; // Not too small\n                    });\n                \n                // Group them into likely conversation turns by analyzing text patterns\n                const allText = textElements.map(el => el.textContent.trim());\n                \n                // Try to identify user vs assistant messages\n                const userPatterns = [\n                    /^you:/i, /^user:/i, /^human:/i, \n                    /^\\s*[A-Z]\\s+/, // Single letter prefix (like Claude\'s "H")\n                    /edit$/i, // Edit button suffix\n                ];\n                \n                const assistantPatterns = [\n                    /^claude:/i, /^assistant:

In [88]:
codeblock_list=[CodeBlock(language=line['language'],code=line['code']) for line in response.json()['messages'][-3]['content']['codeBlocks'] if line['language'] in ['python','bash','sh'] ]

In [93]:
code_exe_result=await LocalCodeExecutor(codeblock_list,env='/oper/work/endian/LLM-Assistant/py310/',filedir='/oper/work/endian/intelligent_agent/page_analyzer')

In [94]:
code_exe_result

"Added to path: /oper/work/endian/intelligent_agent\nSuccessfully imported BrowserSession\n正在查找包含 chatgpt.com 的标签页...\n找到包含 chatgpt.com 的标签页: FB8A15F3DA00BE968FBA2E302E72714A\n当前所有标签页: ['FB8A15F3DA00BE968FBA2E302E72714A', '6B8FB9E909F569A8CAB0476D21BBD064']\n目标标签页句柄: FB8A15F3DA00BE968FBA2E302E72714A\n"

In [37]:
#获取所有标签页：
response = requests.get(
    "http://localhost:8005/tabs",
    headers=user_api_key
)
tabs = response.json()
print(tabs)

[{'tab_id': '46e2a78c-b426-4cac-b736-66cf713beeae', 'provider': 'claude', 'title': 'Quantum CUBO Code Example - Claude', 'url': 'https://claude.ai/chat/02512880-b168-425b-a9d1-0b95b44f092b'}]


In [26]:
import uuid

def generate_api_key():
    """生成一个随机的API密钥"""
    return str(uuid.uuid4())

# 示例使用
new_user_key = generate_api_key()

In [30]:
generate_api_key()

'f16ab7cb-c2d9-4f2a-b2a9-968a0df75385'